In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

In [2]:
CSV_PATH = "product_review_sentiment_3000.csv"
MODEL_PATH = "sentiment_model.joblib"
VECTORIZER_PATH = "tfidf_vectorizer.joblib"


def main():
    # 1. Load data
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["Review", "Sentiment"])

    X = df["Review"].astype(str)
    y = df["Sentiment"].astype(str)

    print("Class distribution:")
    print(y.value_counts(), "\n")

    # 2. Train / test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. TF-IDF feature extraction
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000,
        min_df=2,
    )
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    # 4. Train LinearSVC classifier
    model = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
    model.fit(X_train_tfidf, y_train)

    # 5. Evaluate
    y_pred = model.predict(X_test_tfidf)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    # 6. Save model + vectorizer with Joblib
    joblib.dump(model, MODEL_PATH)
    joblib.dump(vectorizer, VECTORIZER_PATH)
    print(f"\nSaved model to '{MODEL_PATH}'")
    print(f"Saved vectorizer to '{VECTORIZER_PATH}'")


if __name__ == "__main__":
    main()

Class distribution:
Sentiment
Neutral     1000
Positive    1000
Negative    1000
Name: count, dtype: int64 

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       200
     Neutral       1.00      1.00      1.00       200
    Positive       1.00      1.00      1.00       200

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600


Saved model to 'sentiment_model.joblib'
Saved vectorizer to 'tfidf_vectorizer.joblib'
